# 05 – Writing Insights

Making charts is the easy part.

The hard part — and the most valuable skill — is knowing **what to say about them**.

A chart sitting in a Jupyter notebook doesn't communicate anything on its own. The person who ran the analysis needs to read the chart, interpret it, and explain it in plain language.

This notebook teaches you how to do that: how to look at a chart and turn it into a concrete, specific, useful insight.

The skill has two parts:
1. **What to look for** in different types of charts
2. **How to write** what you see — clearly, specifically, and without overstating

## The Anatomy of a Good Insight

A weak insight describes the chart.  
A strong insight describes what the chart *means*.

**Weak:** "The histogram shows that Math scores are between 0 and 100."

**Stronger:** "Math scores are roughly normally distributed, centred around 68, with a few outliers below 15 — suggesting most students are performing at a moderate level, with a small number who may need additional support."

The formula for a strong insight is:

```
[What the data shows]  +  [The specific numbers]  +  [What it implies]
```

You don't always need all three — sometimes the implication is obvious. But you always need at least the first two.

## 1) Reading a Histogram — What to Look For

When you look at a distribution, ask these questions:
1. **Where is the centre?** (mean, median — roughly where are most values?)
2. **How wide is the spread?** (tightly packed or spread out?)
3. **Is it symmetric, or skewed?** (tail to the left or right?)
4. **Are there outliers?** (bars far from the main group)
5. **Is there more than one peak?** (bimodal — suggests two distinct groups)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 200

df = pd.DataFrame({
    "Gender":     rng.choice(["Male","Female"], size=n, p=[0.52, 0.48]),
    "Department": rng.choice(["Science","Arts","Commerce"], size=n, p=[0.4,0.3,0.3]),
    "City":       rng.choice(["Mumbai","Pune","Delhi","Bangalore","Chennai"], size=n),
    "Study_Hours": np.clip(rng.normal(5, 2, n).round(1), 0, 12),
    "Attendance":  np.clip(rng.normal(75, 15, n).round(1), 20, 100),
    "Math":        np.clip(rng.normal(68, 18, n).round(0).astype(int), 0, 100),
    "Science":     np.clip(rng.normal(65, 20, n).round(0).astype(int), 0, 100),
    "English":     np.clip(rng.normal(70, 15, n).round(0).astype(int), 0, 100),
    "Fees_Paid":   rng.choice([True, False], size=n, p=[0.78, 0.22]),
})
for col, frac in [("Study_Hours",0.05),("Attendance",0.03),("Math",0.04),("Science",0.04),("English",0.03)]:
    idx = rng.choice(n, size=int(n * frac), replace=False)
    df.loc[idx, col] = np.nan
df.loc[rng.choice(n, 3, replace=False), "Math"] = rng.integers(0, 15, 3)
df["Total"] = df[["Math","Science","English"]].sum(axis=1)

attendance = df["Attendance"].dropna()

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(attendance, bins=20, kde=True, color="steelblue", edgecolor="white", alpha=0.7, ax=ax)
ax.axvline(attendance.mean(),   color="firebrick",  linestyle="--", linewidth=2,
           label=f"Mean: {attendance.mean():.1f}%")
ax.axvline(attendance.median(), color="darkorange",  linestyle="-",  linewidth=2,
           label=f"Median: {attendance.median():.1f}%")
ax.axvline(75, color="black", linestyle=":", linewidth=1.5, label="75% threshold")
ax.set_title("Student Attendance Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Attendance %")
ax.set_ylabel("Number of Students")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/insight_histogram.png", dpi=120)
plt.close()
print("Chart saved.")

**What the chart tells us — written as insights:**

*"Attendance is approximately normally distributed, centred around 75%. The mean (75.3%) and median (75.6%) are nearly identical, confirming the distribution is roughly symmetric — no strong skew in either direction.*

*The 75% threshold line shows that roughly half the students sit right at the exam eligibility cutoff. A meaningful minority — visible in the left tail — fall below 75%, which puts their exam eligibility at risk.*

*There are no dramatic outliers, but the spread is substantial: the distribution ranges from about 30% to nearly 100%, a 70-point range."*

Notice the pattern:
1. **Shape:** "approximately normally distributed"
2. **Centre:** "centred around 75%, mean and median nearly identical"
3. **What the reference line tells us:** "roughly half sit at the cutoff"
4. **The concerning group:** "a meaningful minority fall below 75%"
5. **Spread:** "30% to nearly 100%, a 70-point range"

Each sentence adds a new piece of information. None of them just redescribes the chart — each one interprets it.

## 2) Reading a Bar Chart — What to Look For

For bar charts, ask:
1. **Which category is highest / lowest?** (name it, state the value)
2. **How large is the gap between the top and bottom?** (is it meaningful or trivial?)
3. **Is the pattern expected or surprising?**
4. **Is any category unexpectedly close to another that you'd expect to be different?**

In [ ]:
dept_avg = df.groupby("Department")[["Math","Science","English"]].mean().round(1)
print(dept_avg)
print()

# Which department does best overall?
dept_total = df.groupby("Department")["Total"].mean().round(1)
print("Average Total by Department:")
print(dept_total.sort_values(ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

depts = dept_avg.index
x = range(len(depts))
width = 0.25

for i, (col, colour) in enumerate(zip(["Math","Science","English"],["steelblue","darkorange","seagreen"])):
    bars = ax.bar([xi + i*width for xi in x], dept_avg[col], width,
                  label=col, color=colour, edgecolor="white")

ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels(depts)
ax.set_title("Average Subject Scores by Department", fontsize=13, fontweight="bold")
ax.set_xlabel("Department")
ax.set_ylabel("Average Score")
ax.legend(title="Subject")
ax.set_ylim(55, 80)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/insight_bar.png", dpi=120)
plt.close()
print("Grouped bar chart saved.")

**What the chart tells us — written as insights:**

*"All three departments perform similarly across subjects, with average scores ranging from approximately 64 to 73. No department dominates across the board.*

*Notably, Science students do not score significantly higher in the Science subject than students from other departments — the difference is less than 3 points. This suggests that departmental specialisation does not strongly predict subject-specific performance in this dataset.*

*English scores are consistently the highest across all three departments, while Science scores are the lowest — a pattern worth investigating further."*

Key moves in this insight:
- **Name the range:** "64 to 73" — gives context
- **Name the surprising finding:** Science students don't outperform in Science
- **State the implication:** "departmental specialisation does not strongly predict..."
- **Notice the cross-department pattern:** English consistently highest, Science consistently lowest
- **Flag for follow-up:** "worth investigating further" — you don't pretend to know why

## 3) Reading a Scatter Plot — What to Look For

For scatter plots, ask:
1. **Is there a clear direction?** (upward/downward/flat)
2. **How tight is the cluster?** (strong relationship or weak one?)
3. **Are there outliers that break the pattern?** (points far from the trend)
4. **Is the relationship linear, or curved?**
5. **Do groups (colours) behave differently?**

In [ ]:
clean = df[["Study_Hours","Total","Department"]].dropna()
r_overall = clean["Study_Hours"].corr(clean["Total"])

fig, ax = plt.subplots(figsize=(8, 5))
colours = {"Science":"steelblue","Arts":"darkorange","Commerce":"seagreen"}

for dept in ["Science","Arts","Commerce"]:
    sub = clean[clean["Department"] == dept]
    ax.scatter(sub["Study_Hours"], sub["Total"],
               color=colours[dept], alpha=0.5, edgecolors="none", s=50, label=dept)

# Overall trend line
z = np.polyfit(clean["Study_Hours"], clean["Total"], 1)
x_line = np.linspace(clean["Study_Hours"].min(), clean["Study_Hours"].max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), color="black", linewidth=2, linestyle="--", label=f"Trend (r={r_overall:.2f})")

ax.set_title("Study Hours vs Total Score by Department", fontsize=13, fontweight="bold")
ax.set_xlabel("Daily Study Hours")
ax.set_ylabel("Total Score")
ax.legend(title="Department", fontsize=9)
ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/insight_scatter.png", dpi=120)
plt.close()
print(f"Scatter saved. Overall r = {r_overall:.3f}")

**What the chart tells us — written as insights:**

*"There is a moderate positive correlation (r = [value]) between daily study hours and total score — students who study more tend to score higher. However, the relationship is far from perfect: the scatter is wide, meaning study hours alone explains only a fraction of the variation in scores.*

*There are students who study 8+ hours and still score below 150, and students who study only 3-4 hours and score above 200. This tells us other factors — prior knowledge, teaching quality, test-taking ability — play significant roles.*

*The three departments show no obvious cluster separation in this scatter plot, suggesting that the study-hours-to-score relationship is consistent across departments."*

Notice what this insight does:
- **States the direction and strength:** "moderate positive... r = [value]"
- **Immediately qualifies it:** "far from perfect: the scatter is wide"
- **Uses specific counter-examples:** "8+ hours... still below 150"
- **Draws the real-world implication:** "other factors play significant roles"
- **Comments on the group pattern:** "no obvious cluster separation"

It never says "studying causes higher scores." It says "tends to score higher" and "associated with" — precise language.

## 4) Reading a Heatmap — What to Look For

For correlation heatmaps, ask:
1. **Which pairs show the strongest positive correlation?** (darkest red)
2. **Which pairs show the weakest correlation?** (closest to white)
3. **Are there any negative correlations?** (blue)
4. **Is the pattern expected or surprising?**

In [ ]:
numeric_cols = ["Study_Hours","Attendance","Math","Science","English","Total"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # hide upper triangle

sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax, mask=mask)

ax.set_title("Correlation Heatmap", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/insight_heatmap.png", dpi=120)
plt.close()
print("Masked heatmap saved.")
print()
print("Top 5 correlations (excluding diagonal):")
# Get upper triangle correlations
corr_pairs = corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool)).stack()
print(corr_pairs.abs().sort_values(ascending=False).head(5).round(3))

**What the chart tells us — written as insights:**

*"The strongest correlations are between the three subject scores — Math, Science, and English correlate with each other at [values], indicating that general academic ability is a significant underlying factor: students who score well in one subject tend to score well across all subjects.*

*Attendance shows a weak-to-moderate positive correlation with scores, confirming that coming to class matters, but it is not the dominant predictor.*

*Study Hours shows the weakest correlation with individual subject scores, but a somewhat stronger link with Total — suggesting study habits have a cumulative rather than subject-specific effect.*

*No strong negative correlations appear in this dataset."*

## 5) How to Avoid Common Insight Mistakes

**Mistake 1: Describing instead of interpreting**

❌ "The bar chart shows three bars of different heights."  
✅ "Science students score highest in Math on average (72.4), trailing Commerce students by only 1.6 points."

**Mistake 2: Overstating causation**

❌ "Students who study more get better grades."  
✅ "Students who study more *tend* to score higher, though the relationship is moderate (r ≈ 0.4) and many other factors influence scores."

**Mistake 3: Ignoring the magnitude**

❌ "Science students score higher than Arts students in Math."  
✅ "Science students score higher than Arts students in Math by an average of 4.3 points — a modest difference given the overall score range of 0–100."

**Mistake 4: Not mentioning what's missing**

Good EDA acknowledges gaps. "This analysis is limited to 200 students. Whether these patterns hold for the full cohort would require a larger dataset."

**Mistake 5: Ignoring outliers**

If you see outliers in a chart, you need to acknowledge them. Even if you can't explain them, say they exist and flag them for further investigation.

## 6) Structuring an EDA Summary

At the end of an EDA, you write a summary. Here's a template:

```
## Key Findings

### Data Overview
- [Number of rows, columns, time period if applicable]
- [Number of missing values and which columns are affected]

### Distribution Insights
- [What the main numeric variables look like: shape, centre, spread]
- [Any unusual distributions or outliers]

### Group Comparisons
- [How the main outcome varies across categories]
- [Which groups differ most / least]

### Relationships
- [Strongest correlations found]
- [Any surprising correlations or lack of correlation]

### Questions for Further Investigation
- [What the EDA raised that it couldn't answer]
```

The last section is just as important as the others. Good EDA generates questions.

In [ ]:
# Generate the numbers needed for a proper summary
print("=== DATA OVERVIEW ===")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()} across {(df.isnull().sum() > 0).sum()} columns")
print()

print("=== SCORE DISTRIBUTIONS ===")
for col in ["Math","Science","English","Total"]:
    clean = df[col].dropna()
    skew = clean.skew()
    skew_label = "right-skewed" if skew > 0.3 else "left-skewed" if skew < -0.3 else "roughly symmetric"
    print(f"{col:10}: mean={clean.mean():.1f}, std={clean.std():.1f}, "
          f"range=[{clean.min():.0f},{clean.max():.0f}], {skew_label}")
print()

print("=== ATTENDANCE ===")
att = df["Attendance"].dropna()
below_75 = (att < 75).sum()
print(f"Mean: {att.mean():.1f}%  |  {below_75} students ({below_75/len(att)*100:.1f}%) below 75% threshold")
print()

print("=== TOP CORRELATIONS (vs Total) ===")
numeric = df[["Study_Hours","Attendance","Math","Science","English","Total"]].dropna()
corr_total = numeric.corr()["Total"].drop("Total").sort_values(key=abs, ascending=False)
for col, r in corr_total.items():
    print(f"  {col:15}: r = {r:+.3f}")

Always generate the exact numbers before writing the summary. Never estimate — "approximately 20%" is weaker than "21.4% (43 students)". The numbers make insights credible and specific.

In [ ]:
# Write the full summary as a formatted print
print("""
EDA SUMMARY — Student Performance Dataset
==========================================

DATA OVERVIEW
 - 200 students, 13 variables covering demographics, habits, and academic scores
 - Missing values are minor: <5% in any column (Study Hours, Attendance, subject scores)
   Imputation with column means is appropriate before any modelling

SCORE DISTRIBUTIONS
 - All three subject scores are roughly normally distributed, centred between 65-70
 - Math has the highest mean (≈68) but also the most outliers — 3 students scored below 15,
   which warrants investigation before modelling
 - Science has the highest variability (std ≈20) compared to English (std ≈15)
 - Total scores cluster around 200 out of 300

ATTENDANCE
 - Attendance averages 75%, the exam eligibility threshold
 - ~{below_75}% of students fall below 75%, meaning a meaningful number are at risk
 - No strong skew — the distribution is symmetric

GROUP COMPARISONS
 - Department has minimal effect on subject scores (differences < 5 points)
 - Science students do NOT outperform in Science vs other departments — counter-intuitive
 - English is consistently the highest-scoring subject across all departments

KEY RELATIONSHIPS
 - Subject scores strongly correlate with each other (r = 0.5-0.7), indicating a strong
   general academic ability factor
 - Attendance has a moderate positive correlation with scores
 - Study hours show a weaker correlation than expected with individual subjects

QUESTIONS FOR FURTHER INVESTIGATION
 - What explains the 3 extreme Math outliers? Data entry errors or genuine poor performance?
 - Why don't Science-department students outperform in Science?
 - With a larger dataset: do these patterns hold across years and cohorts?
""".format(below_75=round((att < 75).sum() / len(att) * 100, 1)))

This is a proper EDA summary — not a list of chart descriptions, but a set of concrete findings with numbers, grouped by theme, and ending with open questions.

Writing this summary is the goal. Everything you learned in the previous notebooks — histograms, boxplots, scatter plots, heatmaps — is in service of producing something like this.

## Summary

- **Good insights = what the data shows + specific numbers + what it implies**
- Match your language to the chart type:
  - Histogram → shape, centre, spread, skew, outliers
  - Bar chart → ranking, gaps, surprising comparisons, cross-category patterns
  - Scatter → direction, strength, outliers, group differences
  - Heatmap → strongest pairs, unexpected weaknesses, any negatives
- **Never overstate causation** — use "associated with", "tends to", "suggests"
- **Always include numbers** — "higher" means nothing; "higher by 4.3 points" means something
- **Structure your summary** — overview, distributions, groups, relationships, open questions
- Good EDA generates questions as well as answers

Next up: **EDA Mini-Project** — a full end-to-end analysis on a new dataset.

## Practice Questions 1

Run the following analyses and write a 3-sentence insight for each result:

1. Plot the distribution of `Study_Hours`. Write one sentence about the shape, one about the centre, and one about any unusual observations.
2. Create a grouped bar chart of average `Attendance` by `Department` and `Gender`. Write what you find.
3. Create a scatter plot of `Math` vs `English`. Write one sentence about direction, one about strength, and one about what it implies.

## Practice Questions 2

Write a complete 4-paragraph EDA summary for the student dataset covering:
1. Data quality (missing values, outliers)
2. Score distributions (are they normal? any skew? surprising ranges?)
3. The strongest relationship you found
4. One open question the EDA raised but couldn't answer